In [1]:
import numpy as np
import pandas as pd
import dai
import structlog

logger = structlog.get_logger()

EMPTY_RESULT = pd.DataFrame(columns=["date", "instrument", "factor"])


def main(datasources, start_date, end_date):
    """
    基于1分钟K线构建12因子复合信号。

    因子：
        f1  VWAP动量
        f2  5日动量
        f3  成交量比
        f4  VWAP偏离
        f5  5日收益率 × LogVolume
        f6  日收益率
        f7  MA5 Bias
        f8  Bollinger偏离
        f9  日内趋势
        f10 尾盘30分钟收益
        f11 分钟实现波动率
        f12 尾盘30分钟成交量占比

    约定：
        当日因子在收盘后生成，用于下一交易日。
    """
    bar_table = datasources.get("bar1m")
    if not bar_table:
        return EMPTY_RESULT.copy()

    start_ts = pd.Timestamp(start_date).normalize()
    end_ts = pd.Timestamp(end_date).normalize()

    if start_ts > end_ts:
        logger.error("开始日期晚于结束日期")
        return EMPTY_RESULT.copy()

    buffer_days = 90
    query_start = start_ts - pd.Timedelta(days=buffer_days)

    output_start = start_ts.strftime("%Y-%m-%d %H:%M:%S")
    output_end = end_ts.strftime("%Y-%m-%d %H:%M:%S")

    sql = f"""
    WITH minute_base AS (
        SELECT
            instrument,
            date,
            CAST(strftime(date, '%Y-%m-%d') AS DATETIME) AS trading_day,
            open,
            high,
            low,
            close,
            volume,
            amount,
            lag(close, 1) OVER (
                PARTITION BY
                    instrument,
                    CAST(strftime(date, '%Y-%m-%d') AS DATETIME)
                ORDER BY date ASC
            ) AS minute_close_lag1
        FROM {bar_table}
        WHERE close > 0
          AND volume >= 0
    ),

    minute_features AS (
        SELECT
            *,
            close / nullif(minute_close_lag1, 0) - 1 AS minute_return,
            row_number() OVER (
                PARTITION BY instrument, trading_day
                ORDER BY date DESC
            ) AS reverse_minute_no
        FROM minute_base
    ),

    daily AS (
        SELECT
            instrument,
            trading_day,

            first(open ORDER BY date ASC) AS open,
            last(close ORDER BY date ASC) AS close,
            max(high) AS high,
            min(low) AS low,

            sum(volume) AS volume,
            sum(amount) AS amount,
            sum(amount) / nullif(sum(volume), 0) AS vwap,

            count(*) AS minute_count,

            sqrt(
                sum(
                    CASE
                        WHEN minute_return IS NOT NULL
                        THEN power(minute_return, 2)
                    END
                )
            ) AS realized_vol,

            (
                last(close ORDER BY date ASC)
                /
                nullif(
                    first(close ORDER BY date ASC)
                        FILTER (WHERE reverse_minute_no <= 30),
                    0
                )
                - 1
            ) AS tail_return,

            sum(volume) FILTER (
                WHERE reverse_minute_no <= 30
            ) / nullif(sum(volume), 0) AS tail_volume_ratio

        FROM minute_features
        GROUP BY instrument, trading_day
    ),

    with_lags AS (
        SELECT
            trading_day,
            instrument,
            open,
            high,
            low,
            close,
            volume,
            amount,
            vwap,
            realized_vol,
            tail_return,
            tail_volume_ratio,
            minute_count,

            lag(vwap, 1) OVER (
                PARTITION BY instrument
                ORDER BY trading_day ASC
            ) AS vwap_lag1,

            lag(close, 1) OVER (
                PARTITION BY instrument
                ORDER BY trading_day ASC
            ) AS close_lag1,

            lag(close, 5) OVER (
                PARTITION BY instrument
                ORDER BY trading_day ASC
            ) AS close_lag5,

            avg(volume) OVER (
                PARTITION BY instrument
                ORDER BY trading_day ASC
                ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
            ) AS volume_avg5,

            avg(close) OVER (
                PARTITION BY instrument
                ORDER BY trading_day ASC
                ROWS BETWEEN 4 PRECEDING AND CURRENT ROW
            ) AS close_avg5,

            avg(close) OVER (
                PARTITION BY instrument
                ORDER BY trading_day ASC
                ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
            ) AS close_avg20,

            stddev_samp(close) OVER (
                PARTITION BY instrument
                ORDER BY trading_day ASC
                ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
            ) AS close_std20

        FROM daily
    ),

    factors AS (
        SELECT
            trading_day AS date,
            instrument,

            -- f1：VWAP动量
            vwap / nullif(vwap_lag1, 0) - 1 AS f1,

            -- f2：5日动量
            close / nullif(close_lag5, 0) - 1 AS f2,

            -- f3：成交量相对过去5日均值
            volume / nullif(volume_avg5, 0) - 1 AS f3,

            -- f4：收盘价相对VWAP偏离
            (close - vwap) / nullif(vwap, 0) AS f4,

            -- f5：5日收益率 × LogVolume
            (
                close / nullif(close_lag5, 0) - 1
            ) * ln(1 + greatest(volume, 0)) AS f5,

            -- f6：日收益率
            close / nullif(close_lag1, 0) - 1 AS f6,

            -- f7：MA5 Bias
            (close - close_avg5) / nullif(close_avg5, 0) AS f7,

            -- f8：Bollinger偏离
            (close - close_avg20)
                / nullif(2 * close_std20, 0) AS f8,

            -- f9：日内趋势
            (close - open) / nullif(high - low, 0) AS f9,

            -- f10：尾盘30根分钟K线收益率
            tail_return AS f10,

            -- f11：分钟实现波动率
            realized_vol AS f11,

            -- f12：尾盘成交量占比
            tail_volume_ratio AS f12

        FROM with_lags
        WHERE close_lag5 IS NOT NULL
          AND volume_avg5 IS NOT NULL
          AND minute_count >= 30
    ),

    stock_pool AS (
        SELECT DISTINCT
            CAST(strftime(date, '%Y-%m-%d') AS DATETIME) AS date,
            CAST(instrument AS VARCHAR) AS instrument
        FROM bigalpha_2026_instruments
    ),

    eligible_factors AS (
        SELECT f.*
        FROM factors f
        INNER JOIN stock_pool p
            ON f.date = p.date
           AND CAST(f.instrument AS VARCHAR) = p.instrument
    ),

    zscored AS (
        SELECT
            date,
            instrument,

            (f1 - avg(f1) OVER (PARTITION BY date))
                / nullif(stddev_samp(f1) OVER (PARTITION BY date), 0) AS z1,

            (f2 - avg(f2) OVER (PARTITION BY date))
                / nullif(stddev_samp(f2) OVER (PARTITION BY date), 0) AS z2,

            (f3 - avg(f3) OVER (PARTITION BY date))
                / nullif(stddev_samp(f3) OVER (PARTITION BY date), 0) AS z3,

            (f4 - avg(f4) OVER (PARTITION BY date))
                / nullif(stddev_samp(f4) OVER (PARTITION BY date), 0) AS z4,

            (f5 - avg(f5) OVER (PARTITION BY date))
                / nullif(stddev_samp(f5) OVER (PARTITION BY date), 0) AS z5,

            (f6 - avg(f6) OVER (PARTITION BY date))
                / nullif(stddev_samp(f6) OVER (PARTITION BY date), 0) AS z6,

            (f7 - avg(f7) OVER (PARTITION BY date))
                / nullif(stddev_samp(f7) OVER (PARTITION BY date), 0) AS z7,

            (f8 - avg(f8) OVER (PARTITION BY date))
                / nullif(stddev_samp(f8) OVER (PARTITION BY date), 0) AS z8,

            (f9 - avg(f9) OVER (PARTITION BY date))
                / nullif(stddev_samp(f9) OVER (PARTITION BY date), 0) AS z9,

            (f10 - avg(f10) OVER (PARTITION BY date))
                / nullif(stddev_samp(f10) OVER (PARTITION BY date), 0) AS z10,

            (f11 - avg(f11) OVER (PARTITION BY date))
                / nullif(stddev_samp(f11) OVER (PARTITION BY date), 0) AS z11,

            (f12 - avg(f12) OVER (PARTITION BY date))
                / nullif(stddev_samp(f12) OVER (PARTITION BY date), 0) AS z12

        FROM eligible_factors
    ),

    raw_score AS (
        SELECT
            date,
            instrument,

            0.12 * coalesce(z1, 0) +
            0.14 * coalesce(z2, 0) +
            0.08 * coalesce(z3, 0) +
            0.10 * coalesce(z4, 0) +
            0.08 * coalesce(z5, 0) +
            0.08 * coalesce(z6, 0) +
            0.10 * coalesce(z7, 0) +
            0.08 * coalesce(z8, 0) +
            0.08 * coalesce(z9, 0) +
            0.07 * coalesce(z10, 0) -
            0.04 * coalesce(z11, 0) +
            0.03 * coalesce(z12, 0) AS raw_score

        FROM zscored
    ),

    final_factor AS (
        SELECT
            date,
            instrument,

            -- 原始分数越大，最终factor越接近1
            percent_rank() OVER (
                PARTITION BY date
                ORDER BY raw_score ASC
            ) AS factor

        FROM raw_score
        WHERE raw_score IS NOT NULL
    )

    SELECT
        date,
        instrument,
        factor
    FROM final_factor
    WHERE date >= CAST('{output_start}' AS DATETIME)
      AND date <= CAST('{output_end}' AS DATETIME)
    """

    try:
        df = dai.query(
            sql,
            filters={
                "date": [
                    query_start.strftime("%Y-%m-%d %H:%M:%S"),
                    end_ts.strftime("%Y-%m-%d 23:59:59"),
                ]
            },
            compression=True,
        ).df()
    except Exception as exc:
        logger.error("因子SQL执行失败", error=str(exc))
        return EMPTY_RESULT.copy()

    if df.empty:
        return EMPTY_RESULT.copy()

    try:
        df["date"] = pd.to_datetime(df["date"]).dt.normalize()
        df["instrument"] = df["instrument"].astype(str)

        df["factor"] = pd.to_numeric(
            df["factor"], errors="coerce"
        ).replace([np.inf, -np.inf], np.nan)

        df = df[
            (df["date"] >= start_ts)
            & (df["date"] <= end_ts)
        ]

        df = (
            df.dropna(subset=["date", "instrument", "factor"])
            .drop_duplicates(
                subset=["date", "instrument"],
                keep="last",
            )
            .sort_values(["date", "instrument"])
            .reset_index(drop=True)
        )

        df["factor"] = df["factor"].clip(0, 1).round(6)

        assert not df.duplicated(
            subset=["date", "instrument"]
        ).any(), "存在重复的 date/instrument"

        assert df["factor"].notna().all(), "factor存在空值"

        assert df["factor"].between(
            0, 1, inclusive="both"
        ).all(), "factor超出[0, 1]区间"

        return df[["date", "instrument", "factor"]]

    except Exception as exc:
        logger.error("因子结果清洗失败", error=str(exc))
        return EMPTY_RESULT.copy()


if __name__ == "__main__":
    from bigmodule import M

    print("=" * 60)
    print("覆盖度自检")
    print("=" * 60)

    test_datasources = {
        "bar1m": "bigalpha_2026_stock_bar1m_selftest"
    }
    test_start = "2024-01-01 00:00:00"
    test_end = "2024-10-31 23:59:59"

    test_factor = main(
        test_datasources,
        test_start,
        test_end,
    )

    if test_factor.empty:
        print("❌ 未生成因子数据")
    else:
        missing_rate = test_factor.groupby("date")["factor"].apply(
            lambda values: values.isna().mean()
        )
        print(f"最大缺失率: {missing_rate.max():.4f}")
        print(
            "✅ 通过"
            if missing_rate.max() <= 0.4
            else "❌ 失败"
        )

    print("\n" + "=" * 60)
    print("未来函数自检")
    print("=" * 60)

    try:
        from lookahead_selfcheck import check_lookahead

        check_start = "2024-01-01 00:00:00"
        check_end = "2024-01-31 23:59:59"

        full = main(
            {"bar1m": "bigalpha_2026_stock_bar1m_selftest"},
            check_start,
            check_end,
        )
        cut = main(
            {"bar1m": "bigalpha_2026_stock_bar1m_ahead"},
            check_start,
            check_end,
        )

        cutoff = pd.Timestamp(check_end) - pd.Timedelta(days=1)

        check_lookahead(
            full,
            cut,
            cutoff=cutoff.strftime("%Y-%m-%d %H:%M:%S"),
        )
        print("✅ 未来函数检测通过")

    except ImportError:
        print("⚠️ 未安装 lookahead_selfcheck")
    except Exception as exc:
        print(f"❌ 未来函数检测异常: {exc}")

    print("\n" + "=" * 60)
    print("官方评估")
    print("=" * 60)

    datasources = {
        "bar1m": "bigalpha_2026_stock_bar1m"
    }
    start_date = "2024-01-01 00:00:00"
    end_date = "2024-12-31 23:59:59"

    factor_data = main(
        datasources,
        start_date,
        end_date,
    )

    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={"date": [start_date, end_date]},
    ).df()

    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )

    print(result)

覆盖度自检
最大缺失率: 0.0000
✅ 通过

未来函数自检
⚠️ 未安装 lookahead_selfcheck

官方评估
